## Import libraries


In [ ]:
import math
import os
import sys
import warnings
from glob import glob

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import rasterio as rio

from hydrokit.input.watershed import get_dem
from hydrokit.utils import create_grids

warnings.filterwarnings("ignore")

## Load the region and create grids


In [ ]:
roi_path = (
    "/beegfs/halder/GITHUB/PROJECT/hydrokit/data/raw/study_area/Bode_catchment.zip"
)
roi = gpd.read_file(roi_path)

# Extract bounding box
bbox = list(roi.iloc[0].geometry.bounds)
minx, miny, maxx, maxy = roi.total_bounds
bbox_rect = Rectangle(
    (minx, miny),            # lower-left corner
    maxx - minx,             # width
    maxy - miny,             # height
    fill=False,
    edgecolor="blue",
    linewidth=1,
    linestyle="--"
)

fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(bbox_rect)
roi.plot(ax=ax, facecolor="none", edgecolor="red")
plt.show()

## Watershed data


### Download and process DEM data


In [ ]:
# Download DEM data
dem_path = "dem_latlon.tif"
dem, meta = get_dem(bbox=bbox, out_path=dem_path, nodata_value=-9999, buffer_ratio=0.5)

### Prepare flow direction and accumulation


In [ ]:
fdir, acc = calculate_flow_metrics(dem_path, "fdir_latlon.tif", "acc_latlon.tif")

# Plot the data
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(4 * 3, 4))
axes = axes.flatten()

for i, arr in enumerate([dem, fdir, acc]):
    axes[i].imshow(arr)

plt.show()

### Prepare basin data


In [ ]:
# Read the basin geometry
basin_path = "/beegfs/halder/GITHUB/PROJECT/hydrokit/data/raw/catchment/WB_dissolved_colorid_.geojson"
basin_gdf = gpd.read_file(basin_path)
basin_gdf.rename(columns={"fid": "basin_id"}, inplace=True)

# Ensure same CRS
basin_gdf = basin_gdf.to_crs(domain_gdf.crs)

# Create centroids
basin_centroids = basin_gdf.copy()
basin_centroids["geometry"] = basin_centroids.geometry.centroid

# Spatial join
basin_with_domain = gpd.sjoin(
    basin_centroids, domain_gdf[["ID", "geometry"]], how="left", predicate="within"
)

# Rename domain ID
basin_with_domain = basin_with_domain.rename(columns={"ID": "domain_id"})

# Join back to original basin polygons
basin_gdf = basin_gdf.join(basin_with_domain["domain_id"])
basin_gdf = basin_gdf[["basin_id", "domain_id", "geometry"]]

print(basin_gdf.shape)
basin_gdf.head()

In [ ]:
basins = get_basin(dem_path, basin_gdf, out_path="basins_latlon.tif", nodata=-9999)
print(basins.shape)

plt.imshow(basins, vmin=0, cmap="tab10")
plt.colorbar()
plt.show()

### Prepare domain mask


In [ ]:
domain_mask = create_domain_mask(
    dem_path, domain_gdf, out_path="mask_org_latlon.tif", nodata=-9999
)
print(domain_mask.shape)

plt.imshow(domain_mask, vmin=0)
plt.show()

## Soil data


In [ ]:
from hydrokit.input import soil

# Download the data
reference_path = "dem_latlon.tif"
out_dir = "soil"
soil.get_soil(reference_path, out_dir)

# Prepare the soil workspace
soil_workspace = soil.SoilWorkspace(reference_path, out_dir)
soil_workspace.build("soil_processed")

In [ ]:
soil_properties = {
    # Texture
    "Clay (%)": "clay",
    "Sand (%)": "sand",
    "Silt (%)": "silt",
    "Quartz (kg/kg)": "qtz",
    "Texture Class": "texture_class",
    # Hydraulic Properties
    "Saturated soil diffusivity (dsat)": "dsat",
    "Saturated Hydraulic Conductivity (ksat)": "ksat",
    "Saturated Soil Water Content (thetas)": "thetas",
    "Residual Soil Water Content (thetar)": "thetar",
    "Field Capacity (theta33)": "theta33",
    "Wilting Point (theta1500)": "theta1500",
    "Pore Size Distribution (bb)": "bb",
    "Bubbling Pressure (hb)": "hb",
    "Pore Size Distribution Index (lambda)": "lambda",
    "Air Entry Pressure (psisat)": "psisat",
    "Organic Matter (om)": "om",
}

for property_name, folder_name in soil_properties.items():
    folder_path = "soil_processed"

    if not os.path.exists(folder_path):
        print(
            f"Skipping {property_name}: Folder '{folder_name}' not found at {folder_path}"
        )
        continue

    depth_files = sorted(glob(os.path.join(folder_path, folder_name, "*.tif")))

    if not depth_files:
        print(f"Skipping {property_name}: No .tif files found in {folder_path}")
        continue

    num_files = len(depth_files)
    cols = 6
    rows = math.ceil(num_files / cols)

    fig, axes = plt.subplots(
        rows, cols, figsize=(15, 5 * rows), constrained_layout=True
    )

    if num_files == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    fig.suptitle(f"{property_name} by Depth", fontsize=12, y=0.8)

    for i, file_path in enumerate(depth_files):
        try:
            with rio.open(file_path) as src:
                data = src.read(1)
                meta = src.meta

                if src.nodata is not None:
                    data = np.ma.masked_equal(data, src.nodata)

                im = axes[i].imshow(data, cmap="viridis")
                axes[i].set_title(os.path.basename(file_path), fontsize=10)
                axes[i].axis("off")

                plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            axes[i].text(0.5, 0.5, "Error reading file", ha="center", va="center")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.show()